# UC2/ImSwitch Example with Real-time Pipeline Processing

This notebook demonstrates how to use the UC2 microscope class to control ImSwitch via HTTP API and process images in real-time using webhooks.

## Features
- Send acquisition schedules (`df_acquire`) to ImSwitch server
- Receive frame notifications via webhook callbacks
- Process images with segmentation, tracking, and feature extraction as they're acquired
- Support for multi-FOV, multi-channel, time-lapse experiments

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Fov, Channel, StimTreatment
from pprint import pprint
import pandas as pd
import numpy as np

## 1. Import Required Libraries

In [3]:
from rtm_pymmcore.microscope.UC2 import UC2
mic = UC2(server_url="http://localhost:8001")  # adjust if needed


## 2. Initialize UC2 Microscope Connection

Connect to the ImSwitch server. The default webhook port is 9000, but you can customize it:
```python
mic = UC2(server_url="http://localhost:8001", webhook_port=9000)
```

In [4]:
## Configuration options - set experiment timing, storage and pipeline parameters
# General timing and frame counts:
N_FRAMES = 2  # number of timesteps

# If you want the notebook/script to wait before starting the experiment, set this (hours).
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

# Timing for acquisition: interval between timesteps (seconds) and approximate time per FOV (seconds).
TIME_BETWEEN_TIMESTEPS = 2  # seconds between timesteps
TIME_PER_FOV = 1  # seconds per FOV (camera + stage moves + overhead)

## Storage path for the experiment - change to your desired directory and experiment name.
base_path = "/tmp"
experiment_name = "exp_test"
path = os.path.join(base_path, experiment_name)


## Channels: images to acquire each timestep.
# Each Channel(...) maps to a microscope channel name configured in Micro-Manager/your device.
# If exposure or power is omitted, the hardware default (set in the device/GUI) will be used.
channels = []
channels = []
channels.append(Channel(name="LASER", exposure=150))
channels.append(Channel(name="LED", exposure=150))

# Experimental condition(s): a list of labels assigned to FOVs.
condition = ["FGFR_high"]  # Example of adding a condition to the dataframe
# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24  # Example: multiple conditions

# If using wellplates, set how many FOVs per well. Set to None if not using wellplates.
n_fovs_per_well = None  ## number of FOVs per well; use None for free-FOV experiments

## Define the Tools that you are using for the experiment (segmentors, trackers, feature extractors)
from rtm_pymmcore.segmentation.base_segmentation import SegmentatorBinary
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.simple_fe import SimpleFE

segmentators = [
    {
        "name": "labels",
        "class": SegmentatorBinary(),  # Note: instantiate the segmentator class
        "use_channel": 0,
        "save_tracked": True,
    },
]

stimulator = None  # No stimulation in this experiment
feature_extractor = SimpleFE("labels")
tracker = TrackerTrackpy()

from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
)
mic.set_pipeline(pipeline=pipeline)

Directory /tmp/exp_test/raw already exists
Directory /tmp/exp_test/tracks already exists
Directory /tmp/exp_test/particles already exists
Directory /tmp/exp_test/labels already exists


## 3. Configure Experiment and Pipeline

Set up acquisition parameters, channels, and the image processing pipeline. The pipeline will automatically receive frames via webhooks during acquisition.

In [5]:
# Generate FOV objects from microscope / viewer (or load from file)
import rtm_pymmcore.utils as utils

# When loading from file, use generate_fov_objects with filename parameter
fovs = utils.generate_fov_objects(mic, filename=os.path.join("", "fovs.json"))

# Generate acquisition dataframe using utils helper (no stimulation)
df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    condition=condition,
)

# Display settings for better readability
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", True)
print(df_acquire.head())

Total Experiment Time: 0.0005555555555555556h
                                          fov_object  fov  fov_x  fov_y  \
0  <rtm_pymmcore.data_structures.Fov object at 0x...    0   0.00    0.0   
1  <rtm_pymmcore.data_structures.Fov object at 0x...    1  20.01    0.0   
2  <rtm_pymmcore.data_structures.Fov object at 0x...    0   0.00    0.0   
3  <rtm_pymmcore.data_structures.Fov object at 0x...    1  20.01    0.0   

   fov_z fov_name  timestep  time  \
0    0.0        0         0     0   
1    0.0        1         0     0   
2    0.0        0         1     2   
3    0.0        1         1     2   

                                            channels      fname  cell_line  
0  ({'name': 'LASER', 'exposure': 150, 'group': N...  000_00000  FGFR_high  
1  ({'name': 'LASER', 'exposure': 150, 'group': N...  001_00000  FGFR_high  
2  ({'name': 'LASER', 'exposure': 150, 'group': N...  000_00001  FGFR_high  
3  ({'name': 'LASER', 'exposure': 150, 'group': N...  001_00001  FGFR_high  


## 4. Generate FOVs and Acquisition Schedule

Load FOV positions and create the acquisition dataframe that UC2 will convert to ImSwitch format.

In [6]:
# Stop any existing webhook server before running experiments
try:
    mic.stop_webhook_server()
    print("Stopped existing webhook server")
except:
    pass

Stopped existing webhook server


In [10]:
# Run experiment with default settings: synchronous + webhook enabled
result = mic.run_experiment(df_acquire)
print(f"Experiment completed. Status: {result.get('status')}")
print(f"Total frames processed via webhook: {mic.frame_count}")

UC2: Webhook configured: http://localhost:9000/frames
UC2: Webhook test passed (status: 200)
Experiment completed. Status: started
Total frames processed via webhook: 4


## 5. Run Experiment with Webhook Integration

The UC2 class supports several modes:

### Default: Synchronous with webhooks (recommended)
```python
result = mic.run_experiment(df_acquire)
```
- Waits for experiment to complete
- Webhook server automatically started
- Pipeline processes each frame as it's saved by ImSwitch
- Server-side storage at ImSwitch's default location

### With custom save directory
```python
result = mic.run_experiment(df_acquire, save_directory="/tmp/uc2_test")
```
- Frames saved to specified directory on ImSwitch server

### Asynchronous mode
```python
result = mic.run_experiment(df_acquire, wait_for_completion=False)
```
- Returns immediately after starting experiment
- Webhook still receives frame notifications
- Use for long experiments where you want control back

### Preview mode
```python
result = mic.run_experiment(df_acquire, do_preview=True)
```
- Returns experiment info without starting acquisition
- Shows total events, channels, time points, etc.

### Disable webhooks (no real-time processing)
```python
result = mic.run_experiment(df_acquire, enable_webhook=False)
```
- Images only saved on server, no local processing

In [11]:
# Asynchronous mode with preview
result = mic.run_experiment(
    df_acquire, 
    wait_for_completion=False, 
    request_timeout=0.8, 
    do_preview=True
)
print(f"Preview: {result.get('preview')}")
print("Experiment started asynchronously, webhook is receiving frames...")

UC2: Webhook configured: http://localhost:9000/frames
UC2: Webhook test passed (status: 200)
Preview: {'total_events': 4, 'channels': ['LASER', 'LED'], 'z_positions': [], 'time_points': [0, 1], 'axis_order': ['t', 'p', 'z', 'c'], 'estimated_duration_minutes': 0.043333333333333335}
Experiment started asynchronously, webhook is receiving frames...


In [15]:
# Synchronous mode with custom save directory
result = mic.run_experiment(
    df_acquire, 
    wait_for_completion=True, 
    request_timeout=6, 
    do_preview=False,
    save_directory="/tmp/uc2_test"
)
print(f"Images saved to: /tmp/uc2_test on ImSwitch server")
print(f"Result: {result}")

UC2: Webhook configured: http://localhost:9000/frames
UC2: Webhook test passed (status: 200)
Images saved to: /tmp/uc2_test on ImSwitch server
Result: {'status': 'started', 'sequence_info': {'total_events': 4, 'channels': ['LASER', 'LED'], 'z_positions': [], 'time_points': [0, 1], 'xy_positions': 0, 'axis_order': ['t', 'p', 'z', 'c'], 'metadata': {}, 'estimated_duration_minutes': 0.043333333333333335}, 'save_directory': '/tmp/uc2_test', 'estimated_duration_minutes': 0.043333333333333335}


## 6. Monitor Results

During webhook-based acquisition:
- Console prints show frame arrivals with metadata (t, p, c, z indices)
- Pipeline processes each frame automatically (segmentation → tracking → feature extraction)
- Results are stored in the pipeline's storage_path
- Final frame count is reported in `mic.frame_count`

### Webhook Server Details
- Runs on `localhost:9000` by default
- Receives POST requests to `/frames` endpoint
- Payload: `{filepath, event_index: {t,p,c,z}, channel}`
- Images loaded with `tifffile` (preferred) or `PIL`
- Metadata enriched from `df_acquire` (FOV coordinates, conditions, etc.)

In [17]:
# Check webhook server status
config = mic.get_webhook_config()
print(f"Webhook configured: {config.get('webhook_url')}")
print(f"Total frames processed: {mic.frame_count}")

Webhook configured: http://localhost:9000/frames
Total frames processed: 20


In [14]:
# Access pipeline results (tracked cells, features, etc.)
# Results are stored in the pipeline's storage_path
import os
results_path = pipeline.storage_path
print(f"Pipeline results stored in: {results_path}")

# List segmentation/tracking outputs
if os.path.exists(results_path):
    for item in os.listdir(results_path):
        print(f"  - {item}")

Pipeline results stored in: /tmp/exp_test
  - .DS_Store
  - labels
  - tracks
  - particles
  - raw


## 7. Cleanup (Optional)

The webhook server is automatically stopped by `mic.post_experiment()`, but you can also stop it manually:

In [ ]:
# Manually stop webhook server if needed
mic.stop_webhook_server()
print("Webhook server stopped")

UC2: Webhook server stopped
Webhook server stopped


### Troubleshooting: Port Already in Use

If you see "Port 9000 is already in use", you have two options:

**Option 1: Kill the existing process**
```python
!lsof -ti:9000 | xargs kill
```

**Option 2: Use a different port**
```python
mic = UC2(server_url="http://localhost:8001", webhook_port=9001)
mic.set_pipeline(pipeline=pipeline)
```